In [ ]:
!pip install transformer-lens datasets torch matplotlib pandas -q

In [ ]:
from datasets import load_dataset
from transformer_lens import HookedTransformer
import torch
import pandas as pd
import matplotlib.pyplot as plt

dataset = load_dataset("azhx/counterfact")
print(dataset)
print(dataset["train"][0])

In [ ]:
def get_counterfact_sample(index=0):
    sample = dataset["train"][index]
    rewrite = sample["requested_rewrite"]

    subject = rewrite["subject"]
    prompt_template = rewrite["prompt"]
    target_true = rewrite["target_true"]["str"]
    target_new = rewrite["target_new"]["str"]

    prompt = prompt_template.format(subject)

    return {
        "subject": subject,
        "prompt": prompt,
        "target_true": target_true,
        "target_new": target_new,
    }

sample = get_counterfact_sample(0)

sample

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = HookedTransformer.from_pretrained(
    "gpt2-small",
    device=device
)

In [ ]:
prompt = sample["prompt"]

tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)

next_token_logits = logits[0, -1]
predicted_token_id = torch.argmax(next_token_logits).item()
predicted_token = model.to_string(predicted_token_id)

print("Prompt:", prompt)
print("True target:", sample["target_true"])
print("New target:", sample["target_new"])
print("Predicted next token:", predicted_token)

In [ ]:
for layer in range(model.cfg.n_layers):
    pattern_name = f"blocks.{layer}.attn.hook_pattern"
    attention_pattern = cache[pattern_name]
    print(f"Layer {layer}: attention shape = {attention_pattern.shape}")

In [ ]:
layer = 0
head = 0

attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().cpu()

token_labels = model.to_str_tokens(prompt)

plt.figure(figsize=(8, 6))
plt.imshow(attention_pattern)
plt.xticks(range(len(token_labels)), token_labels, rotation=90)
plt.yticks(range(len(token_labels)), token_labels)
plt.xlabel("Key tokens")
plt.ylabel("Query tokens")
plt.title(f"Attention Pattern - Layer {layer}, Head {head}")
plt.colorbar()
plt.show()

In [ ]:
# Analyze average attention strength for every layer and head

attention_scores = []

for layer in range(model.cfg.n_layers):
    pattern_name = f"blocks.{layer}.attn.hook_pattern"
    attention_pattern = cache[pattern_name][0]
    # shape: [num_heads, seq_len, seq_len]

    for head in range(model.cfg.n_heads):
        head_attention = attention_pattern[head]

        avg_attention = head_attention.mean().item()
        max_attention = head_attention.max().item()

        attention_scores.append({
            "layer": layer,
            "head": head,
            "average_attention": avg_attention,
            "maximum_attention": max_attention
        })

attention_df = pd.DataFrame(attention_scores)

attention_df_sorted = attention_df.sort_values(
    by="maximum_attention",
    ascending=False
)

attention_df_sorted.head(10)

In [ ]:
top_heads = attention_df_sorted.head(10)

plt.figure(figsize=(10, 5))
plt.bar(
    [f"L{row.layer}-H{row.head}" for _, row in top_heads.iterrows()],
    top_heads["maximum_attention"]
)
plt.xlabel("Layer-Head")
plt.ylabel("Maximum Attention Score")
plt.title("Top 10 Attention Heads by Maximum Attention")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# More meaningful attention analysis:
# Analyze how much the final token attends to previous tokens for each layer-head

final_token_attention_scores = []

for layer in range(model.cfg.n_layers):
    pattern_name = f"blocks.{layer}.attn.hook_pattern"
    attention_pattern = cache[pattern_name][0]
    # shape: [num_heads, seq_len, seq_len]

    for head in range(model.cfg.n_heads):
        head_attention = attention_pattern[head]

        # Attention distribution from the final token to all previous tokens
        final_token_attention = head_attention[-1, :]

        max_score = final_token_attention.max().item()
        entropy = -(final_token_attention * torch.log(final_token_attention + 1e-9)).sum().item()

        final_token_attention_scores.append({
            "layer": layer,
            "head": head,
            "max_final_token_attention": max_score,
            "attention_entropy": entropy
        })

final_attention_df = pd.DataFrame(final_token_attention_scores)

final_attention_df_sorted = final_attention_df.sort_values(
    by="max_final_token_attention",
    ascending=False
)

final_attention_df_sorted.head(10)

In [ ]:
top_final_heads = final_attention_df_sorted.head(10)

labels = [
    f"L{int(row['layer'])}-H{int(row['head'])}"
    for _, row in top_final_heads.iterrows()
]

plt.figure(figsize=(10, 5))
plt.bar(labels, top_final_heads["max_final_token_attention"])

plt.xlabel("Layer-Head")
plt.ylabel("Max Final Token Attention")
plt.title("Top 10 Attention Heads for Final Token Prediction")

plt.xticks(rotation=45)

plt.show()

In [ ]:
# Inspect which token the strongest attention head focuses on

target_layer = 4
target_head = 11

attention_pattern = cache[
    f"blocks.{target_layer}.attn.hook_pattern"
][0, target_head]

token_labels = model.to_str_tokens(prompt)

final_token_attention = attention_pattern[-1].detach().cpu()

most_attended_index = torch.argmax(final_token_attention).item()

print("Prompt Tokens:")
print(token_labels)

print("\nFinal Token Attention Distribution:")
for i, score in enumerate(final_token_attention):
    print(f"Token {i:2d}: {token_labels[i]:>15} -> {score:.4f}")

print("\nMost Attended Token")
print("-------------------")
print("Layer:", target_layer)
print("Head:", target_head)
print("Most attended token:", token_labels[most_attended_index])
print("Attention score:", final_token_attention[most_attended_index].item())

In [ ]:
# Analyze which token each top attention head focuses on

top_heads_analysis = []

top_10_heads = final_attention_df_sorted.head(10)

for _, row in top_10_heads.iterrows():

    layer = int(row["layer"])
    head = int(row["head"])

    attention_pattern = cache[
        f"blocks.{layer}.attn.hook_pattern"
    ][0, head]

    final_token_attention = attention_pattern[-1].detach().cpu()

    most_attended_index = torch.argmax(final_token_attention).item()

    most_attended_token = token_labels[most_attended_index]

    attention_score = final_token_attention[
        most_attended_index
    ].item()

    top_heads_analysis.append({
        "layer": layer,
        "head": head,
        "most_attended_token": most_attended_token,
        "attention_score": attention_score
    })

top_heads_tokens_df = pd.DataFrame(top_heads_analysis)

top_heads_tokens_df

In [ ]:
# Find heads that attend specifically to subject entity tokens

subject = sample["subject"]
subject_token_labels = model.to_str_tokens(subject)

print("Subject:", subject)
print("Subject tokens:", subject_token_labels)
print("Prompt tokens:", token_labels)

# In this example, subject tokens appear inside the prompt tokens.
# We find the indices of subject-related tokens manually by matching subword pieces.

subject_token_indices = []

for i, token in enumerate(token_labels):
    clean_token = token.strip()

    for subject_token in subject_token_labels:
        clean_subject_token = subject_token.strip()

        if clean_subject_token != "" and clean_subject_token in clean_token:
            subject_token_indices.append(i)

subject_token_indices = sorted(list(set(subject_token_indices)))

print("\nDetected subject token indices:", subject_token_indices)
print("Detected subject tokens:", [token_labels[i] for i in subject_token_indices])

In [ ]:
# Remove special token from subject token indices

subject_token_indices = [
    i for i in subject_token_indices
    if token_labels[i] != "<|endoftext|>"
]

print("Clean subject token indices:", subject_token_indices)
print("Clean subject tokens:", [token_labels[i] for i in subject_token_indices])

In [ ]:
# Rank heads by how much final token attends to subject tokens

subject_attention_scores = []

for layer in range(model.cfg.n_layers):
    attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0]

    for head in range(model.cfg.n_heads):
        head_attention = attention_pattern[head]

        final_token_attention = head_attention[-1].detach().cpu()

        subject_attention_sum = final_token_attention[
            subject_token_indices
        ].sum().item()

        subject_attention_scores.append({
            "layer": layer,
            "head": head,
            "subject_attention_sum": subject_attention_sum
        })

subject_attention_df = pd.DataFrame(subject_attention_scores)

subject_attention_df_sorted = subject_attention_df.sort_values(
    by="subject_attention_sum",
    ascending=False
)

subject_attention_df_sorted.head(10)

In [ ]:
top_subject_heads = subject_attention_df_sorted.head(10)

labels = [
    f"L{int(row['layer'])}-H{int(row['head'])}"
    for _, row in top_subject_heads.iterrows()
]

plt.figure(figsize=(10, 5))
plt.bar(labels, top_subject_heads["subject_attention_sum"])
plt.xlabel("Layer-Head")
plt.ylabel("Attention to Subject Tokens")
plt.title("Top 10 Heads Attending to Subject Entity")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Baseline prediction before ablation

def get_next_token_prediction(model, prompt):
    tokens = model.to_tokens(prompt)
    logits = model(tokens)

    next_token_logits = logits[0, -1]
    predicted_token_id = torch.argmax(next_token_logits).item()
    predicted_token = model.to_string(predicted_token_id)

    return predicted_token, predicted_token_id, next_token_logits

baseline_token, baseline_id, baseline_logits = get_next_token_prediction(model, prompt)

print("Prompt:", prompt)
print("Baseline predicted token:", baseline_token)

In [ ]:
# Head ablation experiment: disable Layer 4 Head 11

target_layer = 4
target_head = 11

def ablate_head_hook(value, hook):
    # value shape: [batch, seq_len, n_heads, d_head]
    value[:, :, target_head, :] = 0
    return value

hook_name = f"blocks.{target_layer}.attn.hook_z"

ablated_logits = model.run_with_hooks(
    tokens,
    fwd_hooks=[
        (hook_name, ablate_head_hook)
    ]
)

ablated_next_token_logits = ablated_logits[0, -1]
ablated_predicted_token_id = torch.argmax(ablated_next_token_logits).item()
ablated_predicted_token = model.to_string(ablated_predicted_token_id)

print("Prompt:", prompt)
print("Baseline predicted token:", baseline_token)
print("After ablating L4-H11:", ablated_predicted_token)

In [ ]:
# Ablate multiple subject-focused heads together

top_subject_heads_to_ablate = subject_attention_df_sorted.head(5)

heads_to_ablate = [
    (int(row["layer"]), int(row["head"]))
    for _, row in top_subject_heads_to_ablate.iterrows()
]

print("Heads to ablate:", heads_to_ablate)


In [ ]:
# Create hooks for multiple head ablations

def make_ablation_hook(head_index):
    def ablation_hook(value, hook):
        # value shape: [batch, seq_len, n_heads, d_head]
        value[:, :, head_index, :] = 0
        return value
    return ablation_hook

fwd_hooks = []

for layer, head in heads_to_ablate:
    hook_name = f"blocks.{layer}.attn.hook_z"
    fwd_hooks.append((hook_name, make_ablation_hook(head)))

multi_ablated_logits = model.run_with_hooks(
    tokens,
    fwd_hooks=fwd_hooks
)

multi_ablated_next_token_logits = multi_ablated_logits[0, -1]
multi_ablated_predicted_token_id = torch.argmax(multi_ablated_next_token_logits).item()
multi_ablated_predicted_token = model.to_string(multi_ablated_predicted_token_id)

print("Prompt:", prompt)
print("Baseline predicted token:", baseline_token)
print("After ablating top 5 subject heads:", multi_ablated_predicted_token)

In [ ]:
# Run ablation experiment on multiple CounterFact samples

def get_prediction_text(model, prompt):
    tokens = model.to_tokens(prompt)
    logits = model(tokens)

    next_token_logits = logits[0, -1]
    predicted_token_id = torch.argmax(next_token_logits).item()
    predicted_token = model.to_string(predicted_token_id)

    return predicted_token


def get_subject_token_indices(prompt, subject):
    prompt_tokens = model.to_str_tokens(prompt)
    subject_tokens = model.to_str_tokens(subject)

    indices = []

    for i, token in enumerate(prompt_tokens):
        if token == "<|endoftext|>":
            continue

        clean_token = token.strip()

        for subject_token in subject_tokens:
            if subject_token == "<|endoftext|>":
                continue

            clean_subject_token = subject_token.strip()

            if clean_subject_token != "" and clean_subject_token in clean_token:
                indices.append(i)

    return sorted(list(set(indices)))


def find_top_subject_heads(prompt, subject, top_k=5):
    tokens = model.to_tokens(prompt)
    logits, cache = model.run_with_cache(tokens)

    subject_indices = get_subject_token_indices(prompt, subject)

    scores = []

    for layer in range(model.cfg.n_layers):
        attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"][0]

        for head in range(model.cfg.n_heads):
            final_token_attention = attention_pattern[head][-1].detach().cpu()

            subject_attention_sum = final_token_attention[
                subject_indices
            ].sum().item()

            scores.append({
                "layer": layer,
                "head": head,
                "subject_attention_sum": subject_attention_sum
            })

    df = pd.DataFrame(scores)
    df = df.sort_values(by="subject_attention_sum", ascending=False)

    top_heads = [
        (int(row["layer"]), int(row["head"]))
        for _, row in df.head(top_k).iterrows()
    ]

    return top_heads


def ablate_heads_and_predict(prompt, heads_to_ablate):
    tokens = model.to_tokens(prompt)

    def make_ablation_hook(head_index):
        def ablation_hook(value, hook):
            value[:, :, head_index, :] = 0
            return value
        return ablation_hook

    fwd_hooks = []

    for layer, head in heads_to_ablate:
        hook_name = f"blocks.{layer}.attn.hook_z"
        fwd_hooks.append((hook_name, make_ablation_hook(head)))

    ablated_logits = model.run_with_hooks(
        tokens,
        fwd_hooks=fwd_hooks
    )

    next_token_logits = ablated_logits[0, -1]
    predicted_token_id = torch.argmax(next_token_logits).item()
    predicted_token = model.to_string(predicted_token_id)

    return predicted_token

In [ ]:
multi_sample_results = []

num_samples = 10

for i in range(num_samples):
    sample_i = get_counterfact_sample(i)

    prompt_i = sample_i["prompt"]
    subject_i = sample_i["subject"]

    baseline_pred = get_prediction_text(model, prompt_i)

    top_heads_i = find_top_subject_heads(
        prompt_i,
        subject_i,
        top_k=5
    )

    ablated_pred = ablate_heads_and_predict(
        prompt_i,
        top_heads_i
    )

    changed = baseline_pred != ablated_pred

    multi_sample_results.append({
        "sample_id": i,
        "prompt": prompt_i,
        "baseline_prediction": baseline_pred,
        "ablated_prediction": ablated_pred,
        "changed": changed,
        "ablated_heads": top_heads_i
    })

multi_results_df = pd.DataFrame(multi_sample_results)

multi_results_df

In [ ]:
# Calculate ablation impact statistics

num_changed = multi_results_df["changed"].sum()
total_samples = len(multi_results_df)

change_percentage = (num_changed / total_samples) * 100

print("Ablation Statistics")
print("-------------------")
print(f"Total samples tested: {total_samples}")
print(f"Predictions changed after ablation: {num_changed}")
print(f"Change percentage: {change_percentage:.2f}%")

In [ ]:
# Visualize changed vs unchanged predictions

labels = ["Changed", "Unchanged"]
values = [
    num_changed,
    total_samples - num_changed
]

plt.figure(figsize=(6, 5))
plt.bar(labels, values)

plt.ylabel("Number of Samples")
plt.title("Effect of Attention Head Ablation")

plt.show()

In [ ]:
import os

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Folders created successfully.")

In [ ]:
# Save important result tables

subject_attention_df_sorted.to_csv(
    "outputs/tables/subject_attention_scores.csv",
    index=False
)

top_subject_heads.to_csv(
    "outputs/tables/top_subject_heads.csv",
    index=False
)

multi_results_df.to_csv(
    "outputs/tables/multi_sample_ablation_results.csv",
    index=False
)

print("Tables saved successfully.")

In [ ]:
ablation_summary = pd.DataFrame([
    {
        "total_samples": total_samples,
        "changed_predictions": int(num_changed),
        "unchanged_predictions": int(total_samples - num_changed),
        "change_percentage": change_percentage
    }
])

ablation_summary.to_csv(
    "results/ablation_summary.csv",
    index=False
)

ablation_summary

In [ ]:
# Save ablation effect plot

labels = ["Changed", "Unchanged"]
values = [
    num_changed,
    total_samples - num_changed
]

plt.figure(figsize=(6, 5))
plt.bar(labels, values)
plt.ylabel("Number of Samples")
plt.title("Effect of Subject-Focused Attention Head Ablation")

plt.savefig(
    "outputs/figures/ablation_effect.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Save subject attention heads plot

top_subject_heads = subject_attention_df_sorted.head(10)

labels = [
    f"L{int(row['layer'])}-H{int(row['head'])}"
    for _, row in top_subject_heads.iterrows()
]

plt.figure(figsize=(10, 5))
plt.bar(labels, top_subject_heads["subject_attention_sum"])
plt.xlabel("Layer-Head")
plt.ylabel("Attention to Subject Tokens")
plt.title("Top 10 Heads Attending to Subject Entity")
plt.xticks(rotation=45)

plt.savefig(
    "outputs/figures/subject_attention_heads.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import os

for root, dirs, files in os.walk("outputs"):
    level = root.replace("outputs", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

print("\nResults folder:")
for file in os.listdir("results"):
    print(file)

In [ ]:
report_table = multi_results_df[
    [
        "sample_id",
        "prompt",
        "baseline_prediction",
        "ablated_prediction",
        "changed"
    ]
]

report_table

In [ ]:
# Separate changed and unchanged predictions

changed_cases = multi_results_df[
    multi_results_df["changed"] == True
]

unchanged_cases = multi_results_df[
    multi_results_df["changed"] == False
]

print("Changed Cases")
print("----------------")
display(
    changed_cases[
        [
            "sample_id",
            "prompt",
            "baseline_prediction",
            "ablated_prediction"
        ]
    ]
)

print("\nUnchanged Cases")
print("----------------")
display(
    unchanged_cases[
        [
            "sample_id",
            "prompt",
            "baseline_prediction",
            "ablated_prediction"
        ]
    ]
)